# Deepfake Detection — Weights Quantization Impact
This notebook expects that we already have on google drive:


*   txt2img_images folder containing the images generated through prompt
*   demo_images folder containing the images generated from real ones along with prompt

Below is a visual representation of the folder structure:
```
demo_images/
├── PreSocial/
│   ├── Real/
│   │   └── FORLAB/
│   │       └── images/ (contains real images)
│   └── Fake/
│       ├── StableDiffusion1.5/
│       │   └── images/ (contains images)
│       ├── StableDiffusion3/
│       │   └── images/ (contains images)
│       └── StableDiffusion3.5/
│           └── images/ (contains images)
```

And this is the strcture of the following cells in the notebook

| Cell | Purpose |
|------|----------|
| 1 | Clone repo + install |
| 2 | Mount Drive |
| 3 | Copy weights |
| 4 | Patch detectors (add SD3.5, SDXL, FLUX.1) |
| 5 | Organize txt2img files into folder structure |
| 6 | Run detectors on IMG2IMG |
| 7 | Collect IMG2IMG results |
| 8 | Run detectors on TXT2IMG |
| 9 | Collect TXT2IMG results |
| 10 | Combine all results |
| 11 | Baseline analysis |
| 12 | All improvements |

In [ ]:
# CELL 1: Clone repo and install dependencies
!git clone https://github.com/truebees-ai/Image-Deepfake-Detectors-Public-Library.git
%cd /content/Image-Deepfake-Detectors-Public-Library
!pip install -r requirements.txt -q

Cloning into 'Image-Deepfake-Detectors-Public-Library'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 324 (delta 42), reused 30 (delta 30), pack-reused 237 (from 1)
Receiving objects: 100% (324/324), 23.08 MiB | 13.80 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Filtering content: 100% (4/4), 184.93 MiB | 6.05 MiB/s, done.
/content/Image-Deepfake-Detectors-Public-Library
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# CELL 3: Copy pretrained weights from Drive
import os, shutil

WEIGHTS_DIR = '/content/gdrive/MyDrive/weights'
REPO_ROOT   = '/content/Image-Deepfake-Detectors-Public-Library'

for det in ['CLIP-D', 'NPR', 'R50_nodown']:
    src     = f'{WEIGHTS_DIR}/detectors/{det}/checkpoint/pretrained/weights/best.pt'
    dst_dir = f'{REPO_ROOT}/detectors/{det}/checkpoint/pretrained/weights'
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        shutil.copy2(src, f'{dst_dir}/best.pt')
        print(f'{det}: OK')
    else:
        print(f'{det}: NOT FOUND — check your Drive path')

CLIP-D: OK
NPR: OK
R50_nodown: OK


In [ ]:
# CELL 4: Patch all three detectors to add SD3.5, SDXL, FLUX.1 support
import os

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'

OLD = "'realFORLAB':['FORLAB']"
NEW = ("'realFORLAB':['FORLAB'],\n"
       "        'sd35':['StableDiffusion3.5'],\n"
       "        'sdxl':['StableDiffusionXL'],\n"
       "        'flux':['FLUX.1']")

# Patch sdXL -> sdxl in gen_keys since YAML uses sdXL
OLD2 = "'sdXL':['StableDiffusionXL']"
NEW2 = "'sdXL':['StableDiffusionXL'],\n        'sdxl':['StableDiffusionXL']"

files_to_patch = [
    f'{REPO_ROOT}/detectors/CLIP-D/utils/dataset.py',
    f'{REPO_ROOT}/detectors/R50_nodown/utils/dataset.py',
    f'{REPO_ROOT}/detectors/NPR/data/__init__.py',
]

for path in files_to_patch:
    with open(path) as f:
        content = f.read()
    if 'sd35' in content:
        print(f'Already patched: {path}')
        continue
    if OLD not in content:
        print(f'ERROR: pattern not found in {path}')
        continue
    content = content.replace(OLD, NEW)
    with open(path, 'w') as f:
        f.write(content)
    with open(path) as f:
        check = f.read()
    print(f'{"OK" if "sd35" in check else "FAILED"}: {path}')

OK: /content/Image-Deepfake-Detectors-Public-Library/detectors/CLIP-D/utils/dataset.py
OK: /content/Image-Deepfake-Detectors-Public-Library/detectors/R50_nodown/utils/dataset.py
OK: /content/Image-Deepfake-Detectors-Public-Library/detectors/NPR/data/__init__.py


In [ ]:
# CELL 5: Organize txt2img files into correct folder structure
# Copy real images from img2img dataset
import os, shutil

DRIVE_IMG2IMG = '/content/gdrive/MyDrive/demo_images'
DRIVE_TXT2IMG = '/content/gdrive/MyDrive/txt2img_images'
TXT2IMG_WORK  = '/content/txt2img_structured'  # working copy in Colab

PREFIX_MAP = {
    'sdxl':  'StableDiffusionXL',
    'flux':  'FLUX.1',
    'sd15':  'StableDiffusion1.5',
    'sd3':   'StableDiffusion3',
    'sd35':  'StableDiffusion3.5',
}

real_src = os.path.join(DRIVE_IMG2IMG, 'PreSocial', 'Real')
real_dst = os.path.join(TXT2IMG_WORK, 'PreSocial', 'Real')
if os.path.exists(TXT2IMG_WORK):
    shutil.rmtree(TXT2IMG_WORK)
shutil.copytree(real_src, real_dst)
print(f'Copied real images to {real_dst}')

placed = {}
skipped = []
for fname in sorted(os.listdir(DRIVE_TXT2IMG)):
    if not fname.lower().endswith(('.png','.jpg','.jpeg')):
        continue
    prefix = fname.split('_')[0].lower()
    if prefix not in PREFIX_MAP:
        skipped.append(fname)
        continue
    gen_folder = PREFIX_MAP[prefix]
    dst_dir = os.path.join(TXT2IMG_WORK, 'PreSocial', 'Fake', gen_folder, 'images')
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(os.path.join(DRIVE_TXT2IMG, fname), os.path.join(dst_dir, fname))
    placed[gen_folder] = placed.get(gen_folder, 0) + 1

print('\nPlaced fake images:')
for gen, count in sorted(placed.items()):
    print(f'  {gen}: {count} images')
if skipped:
    print(f'Skipped (unknown prefix): {skipped}')

print('\nFull structure:')
for root, dirs, files in os.walk(TXT2IMG_WORK):
    depth = root.replace(TXT2IMG_WORK, '').count(os.sep)
    indent = '  ' * depth
    imgs = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))]
    if imgs or not files:
        print(f'{indent}{os.path.basename(root)}/  [{len(imgs)} images]')

Copied real images to /content/txt2img_structured/PreSocial/Real

Placed fake images:
  FLUX.1: 9 images
  StableDiffusion1.5: 9 images
  StableDiffusion3: 9 images
  StableDiffusion3.5: 9 images
  StableDiffusionXL: 9 images

Full structure:
txt2img_structured/  [0 images]
  PreSocial/  [0 images]
    Real/  [0 images]
      FORLAB/  [0 images]
        images/  [280 images]
    Fake/  [0 images]
      FLUX.1/  [0 images]
        images/  [9 images]
      StableDiffusionXL/  [0 images]
        images/  [9 images]
      StableDiffusion3/  [0 images]
        images/  [9 images]
      StableDiffusion3.5/  [0 images]
        images/  [9 images]
      StableDiffusion1.5/  [0 images]
        images/  [9 images]


In [ ]:
# CELL 6: Run detectors on IMG2IMG images
import os, json, sys, shutil, argparse, torch

REPO_ROOT     = '/content/Image-Deepfake-Detectors-Public-Library'
DRIVE_IMG2IMG = '/content/gdrive/MyDrive/demo_images'

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
for key in list(sys.modules.keys()):
    if any(x in key for x in ['launcher','dataset','datasets','data']):
        del sys.modules[key]
import launcher

# Symlink img2img images
REPO_DEMO = os.path.join(REPO_ROOT, 'demo_images')
if os.path.islink(REPO_DEMO): os.unlink(REPO_DEMO)
elif os.path.isdir(REPO_DEMO): shutil.rmtree(REPO_DEMO)
os.symlink(DRIVE_IMG2IMG, REPO_DEMO)

def build_split_json(root_path, out_path):
    test_entries = []
    for mod in ['PreSocial', 'Facebook', 'Telegram', 'X']:
        mod_path = os.path.join(root_path, mod)
        if not os.path.isdir(mod_path): continue
        for dirpath, dirnames, filenames in os.walk(mod_path, followlinks=True):
            imgs = [f for f in filenames
                    if os.path.splitext(f)[1].lower() in ['.png','.jpg','.jpeg']]
            if not imgs: continue
            rel   = os.path.relpath(dirpath, mod_path)
            parts = rel.split(os.sep)
            if len(parts) < 3: continue
            gen = parts[1]; sub = parts[2]
            for fname in sorted(imgs):
                test_entries.append(os.path.join(gen, sub, os.path.splitext(fname)[0]))
    unique = sorted(set(test_entries))
    with open(out_path, 'w') as f:
        json.dump({'test': unique}, f, indent=2)
    print(f'Split: {len(unique)} entries')

IMG2IMG_KEYS = 'realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre'
orig_cfg = launcher.load_config
def cfg_img2img(p):
    c = orig_cfg(p); c['testing'] = [IMG2IMG_KEYS]; return c
launcher.run_demo.__globals__['build_demo_split_json'] = build_split_json
launcher.run_demo.__globals__['load_config'] = cfg_img2img

build_split_json(REPO_DEMO, os.path.join(REPO_ROOT, 'split_demo.json'))

print(f'CUDA: {torch.cuda.is_available()}')
for detector in ['CLIP-D', 'NPR', 'R50_nodown']:
    print(f'\n{"+"*50} [IMG2IMG] {detector}')
    launcher.run_demo(argparse.Namespace(
        demo=True, demo_detector=detector, config_dir='configs',
        weights_name=None, detect=False, image=None,
        weights='pretrained', output=None, dry_run=False,
        detector=None, phases='both'))
    print(f'{detector} done.')

Split: 550 entries
CUDA: True

++++++++++++++++++++++++++++++++++++++++++++++++++ [IMG2IMG] CLIP-D
[demo] Running CLIP-D test with args: --name "demo" --task test --device cuda:0 --split_file /content/Image-Deepfake-Detectors-Public-Library/split_demo.json --data_root /content/Image-Deepfake-Detectors-Public-Library/demo_images --data_keys "realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre" --num_threads 8 --arch opencliplinearnext_clipL14commonpool --norm_type clip --resize_size 200 --resize_ratio 1 --resize_prob 0.2 --cmp_qual 65,100 --cmp_prob 0.5 --resizeSize 224
[demo] Completed. Results saved under detectors/<method>/results/demo/<scenario>/results.csv
CLIP-D done.

++++++++++++++++++++++++++++++++++++++++++++++++++ [IMG2IMG] NPR
[demo] Running NPR test with args: --name "demo" --task test --device cuda:0 --split_file /content/Image-Deepfake-Detectors-Public-Library/split_demo.json --data_root /content/Image-Deepfake-Detectors-Public-Library/demo_images --data_keys "realFORLAB:pre&sd15:pr

In [ ]:
# CELL 7: Collect IMG2IMG results
import glob, pandas as pd, os, shutil
from google.colab import files

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'
dfs = []
for csv_path in glob.glob(f'{REPO_ROOT}/detectors/*/results/**/*.csv', recursive=True):
    det = csv_path.replace(REPO_ROOT+'/detectors/','').split('/')[0]
    df  = pd.read_csv(csv_path, header=0, names=['path','score','label'])
    df['path'] = df['path'].str.strip()
    df['detector'] = det
    df['mode'] = 'img2img'
    dfs.append(df)
    print(f'{det}: {len(df)} rows')

if dfs:
    img2img_df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['path','detector'])
    print(f'\nIMG2IMG total: {len(img2img_df)} rows')
    img2img_df.to_csv('/content/results_img2img.csv', index=False)
    files.download('/content/results_img2img.csv')

    # Clear result folders before txt2img run
    for det in ['CLIP-D','NPR','R50_nodown']:
        res = f'{REPO_ROOT}/detectors/{det}/results/demo'
        if os.path.isdir(res):
            shutil.rmtree(res)
            os.makedirs(res)
    print('Result folders cleared for txt2img run.')
else:
    print('No results found — check Cell 6 completed without errors.')

CLIP-D: 550 rows
NPR: 550 rows
R50_nodown: 550 rows

IMG2IMG total: 1650 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Result folders cleared for txt2img run.


In [ ]:
# CELL 8: Run detectors on TXT2IMG images
import os, json, sys, shutil, argparse, torch

REPO_ROOT    = '/content/Image-Deepfake-Detectors-Public-Library'
TXT2IMG_WORK = '/content/txt2img_structured'

os.chdir(REPO_ROOT)
for key in list(sys.modules.keys()):
    if any(x in key for x in ['launcher','dataset','datasets','data']):
        del sys.modules[key]
import launcher

# Symlink to structured txt2img folder
REPO_DEMO = os.path.join(REPO_ROOT, 'demo_images')
if os.path.islink(REPO_DEMO): os.unlink(REPO_DEMO)
elif os.path.isdir(REPO_DEMO): shutil.rmtree(REPO_DEMO)
os.symlink(TXT2IMG_WORK, REPO_DEMO)

total = sum(len([f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))])
            for _,_,files in os.walk(REPO_DEMO, followlinks=True))
print(f'txt2img images visible: {total}')

def build_split_json(root_path, out_path):
    test_entries = []
    for mod in ['PreSocial', 'Facebook', 'Telegram', 'X']:
        mod_path = os.path.join(root_path, mod)
        if not os.path.isdir(mod_path): continue
        for dirpath, dirnames, filenames in os.walk(mod_path, followlinks=True):
            imgs = [f for f in filenames
                    if os.path.splitext(f)[1].lower() in ['.png','.jpg','.jpeg']]
            if not imgs: continue
            rel   = os.path.relpath(dirpath, mod_path)
            parts = rel.split(os.sep)
            if len(parts) < 3: continue
            gen = parts[1]; sub = parts[2]
            for fname in sorted(imgs):
                test_entries.append(os.path.join(gen, sub, os.path.splitext(fname)[0]))
    unique = sorted(set(test_entries))
    with open(out_path, 'w') as f:
        json.dump({'test': unique}, f, indent=2)
    print(f'Split: {len(unique)} entries')

TXT2IMG_KEYS = 'realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre&sdXL:pre&flux:pre'
orig_cfg = launcher.load_config
def cfg_txt2img(p):
    c = orig_cfg(p); c['testing'] = [TXT2IMG_KEYS]; return c
launcher.run_demo.__globals__['build_demo_split_json'] = build_split_json
launcher.run_demo.__globals__['load_config'] = cfg_txt2img

build_split_json(REPO_DEMO, os.path.join(REPO_ROOT, 'split_demo.json'))

print(f'CUDA: {torch.cuda.is_available()}')
for detector in ['CLIP-D', 'NPR', 'R50_nodown']:
    print(f'\n{"+"*50} [TXT2IMG] {detector}')
    launcher.run_demo(argparse.Namespace(
        demo=True, demo_detector=detector, config_dir='configs',
        weights_name=None, detect=False, image=None,
        weights='pretrained', output=None, dry_run=False,
        detector=None, phases='both'))
    print(f'{detector} done.')

txt2img images visible: 325
Split: 325 entries
CUDA: True

++++++++++++++++++++++++++++++++++++++++++++++++++ [TXT2IMG] CLIP-D
[demo] Running CLIP-D test with args: --name "demo" --task test --device cuda:0 --split_file /content/Image-Deepfake-Detectors-Public-Library/split_demo.json --data_root /content/Image-Deepfake-Detectors-Public-Library/demo_images --data_keys "realFORLAB:pre&sd15:pre&sd3:pre&sd35:pre&sdXL:pre&flux:pre" --num_threads 8 --arch opencliplinearnext_clipL14commonpool --norm_type clip --resize_size 200 --resize_ratio 1 --resize_prob 0.2 --cmp_qual 65,100 --cmp_prob 0.5 --resizeSize 224
[demo] Completed. Results saved under detectors/<method>/results/demo/<scenario>/results.csv
CLIP-D done.

++++++++++++++++++++++++++++++++++++++++++++++++++ [TXT2IMG] NPR
[demo] Running NPR test with args: --name "demo" --task test --device cuda:0 --split_file /content/Image-Deepfake-Detectors-Public-Library/split_demo.json --data_root /content/Image-Deepfake-Detectors-Public-Library/d

In [ ]:
# CELL 9: Collect TXT2IMG results
import glob, pandas as pd
from google.colab import files

REPO_ROOT = '/content/Image-Deepfake-Detectors-Public-Library'
dfs = []
for csv_path in glob.glob(f'{REPO_ROOT}/detectors/*/results/**/*.csv', recursive=True):
    det = csv_path.replace(REPO_ROOT+'/detectors/','').split('/')[0]
    df  = pd.read_csv(csv_path, header=0, names=['path','score','label'])
    df['path'] = df['path'].str.strip()
    df['detector'] = det
    df['mode'] = 'txt2img'
    dfs.append(df)
    print(f'{det}: {len(df)} rows')

if dfs:
    txt2img_df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['path','detector'])
    print(f'\nTXT2IMG total: {len(txt2img_df)} rows')
    txt2img_df.to_csv('/content/results_txt2img.csv', index=False)
    files.download('/content/results_txt2img.csv')
else:
    print('No results found — check Cell 8 completed without errors.')

CLIP-D: 325 rows
NPR: 325 rows
R50_nodown: 325 rows

TXT2IMG total: 975 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CELL 10: Combine IMG2IMG + TXT2IMG into one master CSV
import pandas as pd
from google.colab import files

img2img = pd.read_csv('/content/results_img2img.csv')
txt2img = pd.read_csv('/content/results_txt2img.csv')

# Keep only fake rows from txt2img (real images are shared with img2img)
txt2img_fake = txt2img[txt2img['label'] == 1.0].copy()

combined = pd.concat([img2img, txt2img_fake], ignore_index=True)
combined = combined.drop_duplicates(subset=['path','detector'])

print(f'img2img rows:           {len(img2img)}')
print(f'txt2img fake rows:      {len(txt2img_fake)}')
print(f'Combined total:         {len(combined)}')
print('\nRows per mode/detector/label:')
print(combined.groupby(['mode','detector','label']).size().to_string())

combined.to_csv('/content/all_results_full.csv', index=False)
files.download('/content/all_results_full.csv')
print('\nSaved all_results_full.csv')

img2img rows:           1650
txt2img fake rows:      135
Combined total:         1785

Rows per mode/detector/label:
mode     detector    label
img2img  CLIP-D      0.0      280
                     1.0      270
         NPR         0.0      280
                     1.0      270
         R50_nodown  0.0      280
                     1.0      270
txt2img  CLIP-D      1.0       45
         NPR         1.0       45
         R50_nodown  1.0       45


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved all_results_full.csv


In [ ]:
# CELL 11: Baseline analysis
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

df = pd.read_csv('/content/all_results_full.csv')
df.columns = df.columns.str.strip()
df['path'] = df['path'].str.strip()

def extract_generator(p):
    for g in ['StableDiffusion3.5','StableDiffusion3','StableDiffusion1.5',
              'StableDiffusionXL','FLUX.1','FORLAB']:
        if g in p: return g
    return 'unknown'

def extract_quant(p):
    if 'fp4'  in p: return 'fp4'
    if 'fp8'  in p: return 'fp8'
    if 'fp16' in p: return 'fp16'
    return 'real'

df['generator'] = df['path'].apply(extract_generator)
df['quant']     = df['path'].apply(extract_quant)

def get_probs(s): return 1/(1+np.exp(-np.array(s, dtype=float)))
def get_preds(det, s):
    p = get_probs(s)
    return (p>0.5).astype(int) if det=='NPR' else (np.array(s)>0).astype(int)

def print_baseline(data, title):
    print(f'\n{"-"*55}\n{title}\n{"-"*55}')
    for det, g in data.groupby('detector'):
        s = g['score'].values; l = g['label'].values
        pr = get_probs(s); pd_ = get_preds(det, s)
        f = l==1; r = l==0
        tpr = accuracy_score(l[f],pd_[f]) if f.sum() else 0
        tnr = accuracy_score(l[r],pd_[r]) if r.sum() else 0
        auc = roc_auc_score(l,pr) if len(np.unique(l))>1 else 0
        print(f'  {det:12s} TPR={tpr:.3f} TNR={tnr:.3f} AUC={auc:.3f} '
              f'(real={r.sum()} fake={f.sum()})')

print_baseline(df, 'BASELINE — COMBINED')
print_baseline(df[df['mode']=='img2img'], 'BASELINE — IMG2IMG only')
print_baseline(df[df['mode']=='txt2img'], 'BASELINE — TXT2IMG only')

print('\n\nBaseline TPR per generator (combined, fake only):')
for det, dg in df[df['label']==1].groupby('detector'):
    print(f'\n  {det}:')
    for gen, gg in dg.groupby('generator'):
        preds = get_preds(det, gg['score'].values)
        print(f'    {gen:25s} TPR={preds.mean():.3f} n={len(gg)}')

print('\n\nBaseline TPR per quant level (combined, fake only):')
for det, dg in df[df['label']==1].groupby('detector'):
    print(f'\n  {det}:')
    for q in ['fp4','fp8','fp16']:
        qg = dg[dg['quant']==q]
        if not len(qg): continue
        preds = get_preds(det, qg['score'].values)
        print(f'    {q}: TPR={preds.mean():.3f} n={len(qg)}')

print('\n\nBaseline TPR img2img vs txt2img (fake only):')
for det, dg in df[df['label']==1].groupby('detector'):
    print(f'\n  {det}:')
    for mode, mg in dg.groupby('mode'):
        preds = get_preds(det, mg['score'].values)
        print(f'    {mode}: TPR={preds.mean():.3f} n={len(mg)}')


-------------------------------------------------------
BASELINE — COMBINED
-------------------------------------------------------
  CLIP-D       TPR=0.863 TNR=1.000 AUC=0.997 (real=280 fake=315)
  NPR          TPR=0.089 TNR=1.000 AUC=0.791 (real=280 fake=315)
  R50_nodown   TPR=0.194 TNR=0.996 AUC=0.962 (real=280 fake=315)

-------------------------------------------------------
BASELINE — IMG2IMG only
-------------------------------------------------------
  CLIP-D       TPR=0.841 TNR=1.000 AUC=0.996 (real=280 fake=270)
  NPR          TPR=0.056 TNR=1.000 AUC=0.766 (real=280 fake=270)
  R50_nodown   TPR=0.104 TNR=0.996 AUC=0.956 (real=280 fake=270)

-------------------------------------------------------
BASELINE — TXT2IMG only
-------------------------------------------------------
  CLIP-D       TPR=1.000 TNR=0.000 AUC=0.000 (real=0 fake=45)
  NPR          TPR=0.289 TNR=0.000 AUC=0.000 (real=0 fake=45)
  R50_nodown   TPR=0.733 TNR=0.000 AUC=0.000 (real=0 fake=45)


Baseline TPR pe

In [ ]:
# CELL 12: All improvements
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from google.colab import files

df = pd.read_csv('/content/all_results_full.csv')
df.columns = df.columns.str.strip()
df['path'] = df['path'].str.strip()

def extract_generator(p):
    for g in ['StableDiffusion3.5','StableDiffusion3','StableDiffusion1.5',
              'StableDiffusionXL','FLUX.1','FORLAB']:
        if g in p: return g
    return 'unknown'

def extract_quant(p):
    if 'fp4'  in p: return 'fp4'
    if 'fp8'  in p: return 'fp8'
    if 'fp16' in p: return 'fp16'
    return 'real'

df['generator'] = df['path'].apply(extract_generator)
df['quant']     = df['path'].apply(extract_quant)

def get_probs(s): return 1/(1+np.exp(-np.array(s, dtype=float)))

def find_best_threshold(labels, probs):
    best_t, best_ba = 0.5, 0
    for t in np.percentile(probs, np.arange(1,100)):
        preds = (probs>t).astype(int)
        ba = balanced_accuracy_score(labels, preds)
        if ba > best_ba: best_ba, best_t = ba, t
    return best_t, best_ba

def evaluate(labels, preds, probs):
    f=labels==1; r=labels==0
    tpr = accuracy_score(labels[f],preds[f]) if f.sum() else 0
    tnr = accuracy_score(labels[r],preds[r]) if r.sum() else 0
    ba  = balanced_accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs) if len(np.unique(labels))>1 else 0
    return tpr, tnr, ba, auc

available = [d for d in ['CLIP-D','NPR','R50_nodown'] if d in df['detector'].unique()]

# ============================================================
# Threshold calibration (proper cal/test split)
# ============================================================
print('='*60)
print('THRESHOLD CALIBRATION (cal/test split)')
print('='*60)

cal_splits = {}; test_splits = {}; calibrated_thresholds = {}

for dataset_label, data in [('COMBINED', df),
                              ('IMG2IMG',  df[df['mode']=='img2img']),
                              ('TXT2IMG',  df[df['mode']=='txt2img'])]:
    print(f'\n  --- {dataset_label} ---')
    for det, g in data.groupby('detector'):
        if len(g['label'].unique()) < 2: continue
        cal, test = train_test_split(g, test_size=0.5, stratify=g['label'], random_state=42)
        if dataset_label == 'COMBINED':
            cal_splits[det]  = cal
            test_splits[det] = test
        cal_probs  = get_probs(cal['score'].values)
        best_t, _  = find_best_threshold(cal['label'].values, cal_probs)
        if dataset_label == 'COMBINED':
            calibrated_thresholds[det] = best_t
        test_probs = get_probs(test['score'].values)
        test_preds = (test_probs > best_t).astype(int)
        tpr,tnr,ba,auc = evaluate(test['label'].values, test_preds, test_probs)
        print(f'    {det:12s} threshold={best_t:.4f} TPR={tpr:.3f} TNR={tnr:.3f} BalAcc={ba:.3f} AUC={auc:.3f}')

# ============================================================
# Normalized weighted ensemble (no leakage)
# ============================================================
print('\n'+'='*60)
print('NORMALIZED WEIGHTED ENSEMBLE')
print('='*60)

def build_pivot(data):
    p = data.pivot_table(index='path', columns='detector', values='score', aggfunc='first')
    meta = data.drop_duplicates('path').set_index('path')
    for col in ['label','quant','generator','mode']:
        if col in meta.columns: p[col] = meta[col]
    return p.dropna(subset=available+['label'])

cal_pivot  = build_pivot(pd.concat([cal_splits[d]  for d in available]))
test_pivot = build_pivot(pd.concat([test_splits[d] for d in available]))

auc_weights = {}
for det in available:
    mu = cal_pivot[det].mean(); sigma = cal_pivot[det].std()+1e-8
    cal_pivot[f'norm_{det}']  = (cal_pivot[det]  - mu) / sigma
    test_pivot[f'norm_{det}'] = (test_pivot[det] - mu) / sigma
    cal_pivot[f'prob_{det}']  = get_probs(cal_pivot[f'norm_{det}'].values)
    test_pivot[f'prob_{det}'] = get_probs(test_pivot[f'norm_{det}'].values)
    auc_weights[det] = roc_auc_score(cal_pivot['label'].values,
                                     cal_pivot[f'prob_{det}'].values)

total_auc = sum(auc_weights.values())
print(f'  AUC weights (cal split): { {k:round(v/total_auc,3) for k,v in auc_weights.items()} }')

for piv in [cal_pivot, test_pivot]:
    piv['ensemble_simple']   = piv[[f'prob_{d}' for d in available]].mean(axis=1)
    piv['ensemble_weighted'] = sum((auc_weights[d]/total_auc)*piv[f'prob_{d}'] for d in available)

ensemble_thresholds = {}
for name, col in [('Simple','ensemble_simple'),('AUC-weighted','ensemble_weighted')]:
    best_t,_ = find_best_threshold(cal_pivot['label'].values, cal_pivot[col].values)
    ensemble_thresholds[col] = best_t
    tp = (test_pivot[col].values > best_t).astype(int)
    tpr,tnr,ba,auc = evaluate(test_pivot['label'].values, tp, test_pivot[col].values)
    print(f'  {name:20s} threshold={best_t:.4f} TPR={tpr:.3f} TNR={tnr:.3f} BalAcc={ba:.3f} AUC={auc:.3f}')

best_col = 'ensemble_weighted'
best_t   = ensemble_thresholds[best_col]

print('\n  Ensemble TPR per quant (test split):')
for q in ['fp4','fp8','fp16']:
    m = (test_pivot['quant']==q)&(test_pivot['label']==1)
    if not m.sum(): continue
    print(f'    {q}: TPR={(test_pivot.loc[m,best_col].values>best_t).mean():.3f} n={m.sum()}')

print('\n  Ensemble TPR per generator (test split):')
for gen in sorted(test_pivot['generator'].unique()):
    m = (test_pivot['generator']==gen)&(test_pivot['label']==1)
    if not m.sum(): continue
    print(f'    {gen:25s}: TPR={(test_pivot.loc[m,best_col].values>best_t).mean():.3f} n={m.sum()}')

print('\n  Ensemble TPR img2img vs txt2img (test split):')
for mode in ['img2img','txt2img']:
    m = (test_pivot['mode']==mode)&(test_pivot['label']==1)
    if not m.sum(): continue
    print(f'    {mode}: TPR={(test_pivot.loc[m,best_col].values>best_t).mean():.3f} n={m.sum()}')

# ============================================================
# Bootstrap CIs (test split only)
# ============================================================
print('\n'+'='*60)
print('BOOTSTRAP CONFIDENCE INTERVALS (95%, test split)')
print('='*60)

N_BOOT = 1000
rng = np.random.default_rng(42)

def bootstrap_metrics(labels, probs, threshold):
    aucs,tprs,bas = [],[],[]
    n = len(labels)
    for _ in range(N_BOOT):
        idx = rng.integers(0,n,n)
        l,p = labels[idx], probs[idx]
        if len(np.unique(l))<2: continue
        preds = (p>threshold).astype(int)
        aucs.append(roc_auc_score(l,p))
        f=l==1
        tprs.append(accuracy_score(l[f],preds[f]) if f.sum() else 0)
        bas.append(balanced_accuracy_score(l,preds))
    return np.array(aucs), np.array(tprs), np.array(bas)

test_labels = test_pivot['label'].values
cal_thresholds_norm = {}
for det in available:
    t,_ = find_best_threshold(cal_pivot['label'].values, cal_pivot[f'prob_{det}'].values)
    cal_thresholds_norm[det] = t

for det in available:
    aucs,tprs,bas = bootstrap_metrics(test_labels, test_pivot[f'prob_{det}'].values,
                                      cal_thresholds_norm[det])
    print(f'  {det:12s}')
    print(f'    AUC:    {np.mean(aucs):.3f} [{np.percentile(aucs,2.5):.3f}, {np.percentile(aucs,97.5):.3f}]')
    print(f'    TPR:    {np.mean(tprs):.3f} [{np.percentile(tprs,2.5):.3f}, {np.percentile(tprs,97.5):.3f}]')
    print(f'    BalAcc: {np.mean(bas):.3f} [{np.percentile(bas,2.5):.3f}, {np.percentile(bas,97.5):.3f}]')

best_col = 'ensemble_weighted'
best_t   = ensemble_thresholds[best_col]
aucs,tprs,bas = bootstrap_metrics(test_labels, test_pivot[best_col].values, best_t)
print(f'  Ensemble (weighted):')
print(f'    AUC:    {np.mean(aucs):.3f} [{np.percentile(aucs,2.5):.3f}, {np.percentile(aucs,97.5):.3f}]')
print(f'    TPR:    {np.mean(tprs):.3f} [{np.percentile(tprs,2.5):.3f}, {np.percentile(tprs,97.5):.3f}]')
print(f'    BalAcc: {np.mean(bas):.3f} [{np.percentile(bas,2.5):.3f}, {np.percentile(bas,97.5):.3f}]')

# ============================================================
# Threshold transfer
# ============================================================
print('\n'+'='*60)
print('THRESHOLD TRANSFER')
print('Calibrate on one generator, evaluate on others')
print('='*60)

generators = [g for g in ['StableDiffusion1.5','StableDiffusion3',
                           'StableDiffusion3.5','StableDiffusionXL','FLUX.1']
              if g in df['generator'].unique()]
real_df = df[df['label']==0.0]

for det in available:
    print(f'\n  {det}:')
    for cal_gen in generators:
        cal_fake = df[(df['detector']==det)&(df['generator']==cal_gen)]
        cal_real = real_df[real_df['detector']==det]
        cal_set  = pd.concat([cal_fake, cal_real])
        if len(cal_set['label'].unique())<2: continue
        best_t2,_ = find_best_threshold(cal_set['label'].values,
                                        get_probs(cal_set['score'].values))
        test_gens = [g for g in generators if g!=cal_gen]
        row = f'    Cal={cal_gen[:18]:18s} -> '
        for tg in test_gens:
            ts = df[(df['detector']==det)&(df['generator']==tg)]
            if not len(ts): continue
            tp = (get_probs(ts['score'].values)>best_t2).astype(int)
            row += f'{tg[:6]}={tp.mean():.3f}  '
        print(row)

# Save
test_pivot.to_csv('/content/ensemble_results.csv')
files.download('/content/ensemble_results.csv')
print('\nAll done. Results saved.')

IMPROVEMENT A: THRESHOLD CALIBRATION (cal/test split)

  --- COMBINED ---
    CLIP-D       threshold=0.0151 TPR=1.000 TNR=0.886 BalAcc=0.943 AUC=0.999
    NPR          threshold=0.0000 TPR=0.766 TNR=0.621 BalAcc=0.694 AUC=0.779
    R50_nodown   threshold=0.0006 TPR=0.962 TNR=0.836 BalAcc=0.899 AUC=0.953

  --- IMG2IMG ---
    CLIP-D       threshold=0.0202 TPR=0.985 TNR=0.914 BalAcc=0.950 AUC=0.997
    NPR          threshold=0.0000 TPR=0.696 TNR=0.629 BalAcc=0.662 AUC=0.749
    R50_nodown   threshold=0.0007 TPR=0.941 TNR=0.836 BalAcc=0.888 AUC=0.944

  --- TXT2IMG ---

IMPROVEMENT B+C: NORMALIZED WEIGHTED ENSEMBLE
  AUC weights (cal split): {'CLIP-D': np.float64(0.36), 'NPR': np.float64(0.29), 'R50_nodown': np.float64(0.351)}
  Simple               threshold=0.4923 TPR=0.962 TNR=0.943 BalAcc=0.952 AUC=0.994
  AUC-weighted         threshold=0.4679 TPR=0.981 TNR=0.907 BalAcc=0.944 AUC=0.996

  Ensemble TPR per quant (test split):
    fp4: TPR=0.982 n=57
    fp8: TPR=0.980 n=51
    fp16: T

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All done. Results saved.
